In [1]:
import numpy as np
import pandas as pd

In [2]:
csv_path = "../data/real_bitcoin_blocks_raw.csv"
df = pd.read_csv(csv_path)

df.head()

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.0,0.0,1,1.0
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.0,0.0,1,1.0
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.0,0.0,1,1.0
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.0,0.0,1,1.0
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.0,0.0,1,1.0


In [3]:
# Check where values equal 0, then check across rows
rows_with_zero = df[(df[['total_output_satoshis']] == 0).any(axis=1)]
rows_with_zero

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
501726,501726,2017-12-30 12:55:20+00:00,200,1,18009645,501726,0.0,0.0,0.0,1,1.873105e+12


In [4]:
for col in ["total_output_satoshis", "total_output_satoshis_excl_coinbase"]:
    zero_volume = df[df[col] == 0]
    print(col, "zero-volume blocks:", len(zero_volume))
    print(zero_volume["number"].describe())  # are they clustered early, or scattered?

total_output_satoshis zero-volume blocks: 1
count         1.0
mean     501726.0
std           NaN
min      501726.0
25%      501726.0
50%      501726.0
75%      501726.0
max      501726.0
Name: number, dtype: float64
total_output_satoshis_excl_coinbase zero-volume blocks: 89548
count     89548.000000
mean      72387.225075
std      105262.128796
min           0.000000
25%       22501.750000
50%       45156.500000
75%       75721.250000
max      810811.000000
Name: number, dtype: float64


In [5]:
def clean_zero_volume(df, column, patch=True):
    """
    patch=True  -> replace any 0 values in `column` with the column median (like Ethan's missing-value handling)
    patch=False -> leave the data exactly as it is now (current behavior)
    """
    df_cleaned = df.copy()
    if patch:
        df_cleaned[column] = df_cleaned[column].replace(0, np.nan)
        df_cleaned[column] = df_cleaned[column].fillna(df_cleaned[column].median())
        print(f"Patched {(df[column] == 0).sum()} zero-value row(s) in '{column}'")
    return df_cleaned

In [6]:
df = clean_zero_volume(df=df, column='total_output_satoshis', patch=True)

Patched 1 zero-value row(s) in 'total_output_satoshis'


In [7]:
# Either "total_output_satoshis" or "total_output_satoshis_excl_coinbase"
# Either "full" (all 9 features) or "reduced" (drop avg_transactions and avg_volume)

# Definition of Efficiency
# Either ratio_of_means: (avg_transaction / avg_volume)
# Or     mean_of_ratios: average of (n_transactions / transaction_volume) per block
def preprocess(df, volume_column, efficiency_definition):
    # Rename Columns
    processed_df = df.rename(columns={
        'number': 'block_id',
        'transaction_count': 'n_transactions',
        volume_column: 'transaction_volume',
    })

    # Drop Unwanted Columns
    if volume_column == "total_output_satoshis":
        drop_columns = ['total_output_satoshis_excl_coinbase', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
        
    elif volume_column == "total_output_satoshis_excl_coinbase":
        drop_columns = ['total_output_satoshis', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
    
    processed_df = processed_df.drop(columns=drop_columns)

    # Convert satoshis to BTC and rename
    satoshi_columns = ['transaction_volume']
    processed_df[satoshi_columns] = processed_df[satoshi_columns] / 1e8
    
    # Add Efficiency Defined with ratio of n transactions to volume per block
    #processed_df['block_efficiency'] = processed_df['n_transactions'] / processed_df['transaction_volume']
    processed_df['block_efficiency'] = np.where(
    processed_df['transaction_volume'] > 0,
    processed_df['n_transactions'] / processed_df['transaction_volume'],
    np.nan
)

    # Round-robin miner assignment
    processed_df['miner_id'] = processed_df['block_id'] % 100

    # Calculate fee proxy as n_transactions * transaction_volume
    processed_df['fee_proxy'] = processed_df['n_transactions'] * processed_df['transaction_volume']
    

    # Convert the timestamp column from string to proper datetime
    processed_df['timestamp'] = pd.to_datetime(processed_df['timestamp'])
    
    
    # Groupby miner_id
    miner_df = processed_df.groupby('miner_id').agg(
        blocks_mined=('block_id', 'count'),
        avg_transactions=('n_transactions', 'mean'),
        avg_volume=('transaction_volume', 'mean'),
        avg_fee=('fee_proxy', 'mean'),
        fee_volatility=('fee_proxy', 'std'),
        avg_block_size=('size', 'mean'),
        difficulty=('difficulty', 'mean'),
        efficiency=('block_efficiency', 'mean'), # use mean of ratios as default, average of (n transactions / transaction volume per individual block)
        profitability=('fee_proxy', 'sum'),
        last_block_id=('block_id', 'max'),   # last block a mined by a miner
        last_block_time=('timestamp', 'max') # last block mined timestamp
    ).reset_index()
    
    # convert profitability from total sum to avg profitability
    miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
    miner_df['age'] = processed_df['timestamp'].max() - miner_df['last_block_time']
    
    # Convert the duration into a plain number (float of seconds)
    miner_df['age'] = miner_df['age'].dt.total_seconds()
    
    # Calculate Efficiency Based on Definition of Efficiency
    if efficiency_definition == "ratio_of_means":
        # added very small number (1 * 10^-9) to prevent division by zero
        # if any miner has avg_volume of 0
        miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
    elif efficiency_definition == "mean_of_ratios":
        # use mean of ratios as default, average of (n transactions / transaction volume per individual block)
        pass
    
    
    
    # Calculate median efficiency
    median_efficiency = miner_df['efficiency'].median()
    # print(f"Median Efficiency = {median_efficiency}")
    # label miner as 1 if its efficiency is more than median, else 0
    miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
    
    return miner_df

In [8]:
miner_df = preprocess(df, "total_output_satoshis", "mean_of_ratios")
miner_df.sort_values(by='last_block_time', ascending=False).head(10)

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,efficiency,profitability,last_block_id,last_block_time,age,label
8,8,8110,1111.768187,10136.224152,1.780766e+07,5.113957e+07,635394.452898,7.823222e+12,0.287507,1.780547e+07,810908,2023-10-06 13:37:21+00:00,0.0,0
7,7,8110,1116.194945,10701.430329,1.872428e+07,6.977861e+07,636561.353514,7.823309e+12,0.422076,1.872197e+07,810907,2023-10-06 13:37:00+00:00,21.0,1
6,6,8110,1117.373490,9933.392749,1.811877e+07,5.012387e+07,636283.305795,7.823309e+12,0.290385,1.811654e+07,810906,2023-10-06 12:50:15+00:00,2826.0,0
5,5,8110,1117.982244,10802.388056,1.963532e+07,7.151489e+07,636626.161652,7.823309e+12,0.284028,1.963290e+07,810905,2023-10-06 12:45:49+00:00,3092.0,0
4,4,8110,1123.225524,11427.595817,2.072525e+07,1.492980e+08,633191.847965,7.823309e+12,0.411655,2.072270e+07,810904,2023-10-06 12:36:45+00:00,3636.0,1
3,3,8110,1099.984957,10317.032438,1.838393e+07,5.959257e+07,631024.890999,7.822986e+12,0.366910,1.838166e+07,810903,2023-10-06 12:27:41+00:00,4180.0,1
2,2,8110,1120.462762,10749.633443,1.934361e+07,9.517518e+07,632985.547472,7.822986e+12,0.386658,1.934123e+07,810902,2023-10-06 12:15:02+00:00,4939.0,1
1,1,8110,1113.248212,10876.172794,1.882207e+07,5.546725e+07,633592.245993,7.822986e+12,0.310413,1.881975e+07,810901,2023-10-06 12:01:05+00:00,5776.0,0
0,0,8110,1112.876079,10848.979162,1.910743e+07,7.302996e+07,636997.443157,7.822986e+12,0.351540,1.910507e+07,810900,2023-10-06 11:54:39+00:00,6162.0,1
99,99,8109,1123.059440,11095.425626,1.924070e+07,6.450264e+07,639058.787520,7.824585e+12,0.351457,1.923833e+07,810899,2023-10-06 11:54:09+00:00,6192.0,1


In [9]:
# Train and Test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

def train_and_test(miner_df, seed, feature_set):
    
    feature_columns = []
    if feature_set == "full":
        feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                        'avg_fee', 'fee_volatility', 'avg_block_size', 
                        'difficulty', 'profitability', 'age']
    elif feature_set == "reduced":
        feature_columns = ['blocks_mined', 
                           'avg_fee', 'fee_volatility', 'avg_block_size', 
                           'difficulty', 'profitability', 'age']

    X = miner_df[feature_columns]
    y = miner_df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

    # Scale data using normalization so that different features
    # with huge numbers and small numbers have equal importance

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8), 
        activation='relu', 
        solver='adam', 
        alpha=0.001, 
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        max_iter=100,
        batch_size=16
        )

    # print(model)

    # Fit the model (Training)
    model.fit(X_train_scaled, y_train)
    
    # Test the accuracy of the model
    # Ethan's accuracy:
    # Test Accuracy = 95%
    # 5-fold Cross Validation Accuracy = 48.75%
    y_prediction = model.predict(X_test_scaled)
    # print(f"y prediction: {y_prediction}")
    
    test_accuracy = accuracy_score(y_test, y_prediction)
    # print(test_accuracy)

    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    cv_mean = cv_scores.mean()
    # print(f"Cross-Validation Scores:     {cv_scores}")
    # print(f"Cross-Validation Mean Score: {cv_mean}")
    
    median_efficiency = miner_df['efficiency'].median()
    min_efficiency = miner_df['efficiency'].min()
    max_efficiency = miner_df['efficiency'].max()
    
    
    # High cv score found? should be closer to Ethan's Accuracy
    # corr_avg_volume_efficiency = miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['efficiency'])
    corr_avg_transaction_efficiency = miner_df['avg_transactions'].corr(miner_df['efficiency'])
    corr_avg_volume_efficiency = miner_df['avg_volume'].corr(miner_df['efficiency'])
    
    # print(corr_avg_transaction_efficiency)
    # print(corr_avg_volume_efficiency)
    
    return dict(
        test_accuracy=test_accuracy, cv_mean=cv_mean, 
        median_efficiency=median_efficiency, min_efficiency=min_efficiency, max_efficiency=max_efficiency,
        corr_avg_transaction_efficiency=corr_avg_transaction_efficiency, corr_avg_volume_efficiency=corr_avg_volume_efficiency,
        model_iterations=model.n_iter_
        )

In [10]:
train_and_test(miner_df, 42, "full")

{'test_accuracy': 0.55,
 'cv_mean': np.float64(0.4875),
 'median_efficiency': np.float64(0.3181217328568209),
 'min_efficiency': np.float64(0.2735800481872434),
 'max_efficiency': np.float64(0.45054328283273865),
 'corr_avg_transaction_efficiency': np.float64(0.11250176535326101),
 'corr_avg_volume_efficiency': np.float64(0.03423481197431949),
 'model_iterations': 33}

In [11]:
def preprocess_and_test(df, seed, volume_column, efficiency_definition, feature_set):
    miner_df = preprocess(df, volume_column, efficiency_definition)
    results = train_and_test(miner_df, seed, feature_set)
    full_results = {
        "seed": seed,
        "feature_set": feature_set,
        "volume_column": volume_column,
        **results
    }
    
    return pd.DataFrame([full_results])

In [12]:
results_df = []

# Ratio of Means
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "ratio_of_means", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "ratio_of_means", "reduced"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "ratio_of_means", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "ratio_of_means", "reduced"))

# Mean of Ratios
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "reduced"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "mean_of_ratios", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "mean_of_ratios", "reduced"))

final_results = pd.concat(results_df, ignore_index=True)
final_results

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,42,full,total_output_satoshis_excl_coinbase,0.80,0.8500,0.107405,0.098501,0.114919,-0.000700,-0.982983,25
1,42,reduced,total_output_satoshis_excl_coinbase,0.85,0.7000,0.107405,0.098501,0.114919,-0.000700,-0.982983,37
2,42,full,total_output_satoshis,0.80,0.8500,0.107153,0.098291,0.114630,-0.000307,-0.982917,25
3,42,reduced,total_output_satoshis,0.85,0.7000,0.107153,0.098291,0.114630,-0.000307,-0.982917,37
4,42,full,total_output_satoshis_excl_coinbase,0.45,0.4875,0.575491,0.417039,27766.783956,0.062137,0.094429,39
5,42,reduced,total_output_satoshis_excl_coinbase,0.45,0.4875,0.575491,0.417039,27766.783956,0.062137,0.094429,58
6,42,full,total_output_satoshis,0.55,0.4875,0.318122,0.273580,0.450543,0.112502,0.034235,33
7,42,reduced,total_output_satoshis,0.65,0.5000,0.318122,0.273580,0.450543,0.112502,0.034235,43


In [13]:
seeds = [1, 3, 15, 17, 25, 29, 30, 36, 42, 50, 51, 67, 100]
diff_seed_results_df = []
for seed in seeds:
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "ratio_of_means", "full"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "ratio_of_means", "reduced"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis", "ratio_of_means", "full"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis", "ratio_of_means", "reduced"))

diff_seed_final_results = pd.concat(diff_seed_results_df, ignore_index=True)
diff_seed_final_results

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,1,full,total_output_satoshis_excl_coinbase,0.70,0.8000,0.107405,0.098501,0.114919,-0.000700,-0.982983,28
1,1,reduced,total_output_satoshis_excl_coinbase,0.70,0.6500,0.107405,0.098501,0.114919,-0.000700,-0.982983,60
2,1,full,total_output_satoshis,0.70,0.8000,0.107153,0.098291,0.114630,-0.000307,-0.982917,28
3,1,reduced,total_output_satoshis,0.70,0.6500,0.107153,0.098291,0.114630,-0.000307,-0.982917,60
4,3,full,total_output_satoshis_excl_coinbase,0.85,0.8500,0.107405,0.098501,0.114919,-0.000700,-0.982983,38
5,3,reduced,total_output_satoshis_excl_coinbase,0.80,0.6500,0.107405,0.098501,0.114919,-0.000700,-0.982983,32
6,3,full,total_output_satoshis,0.85,0.8500,0.107153,0.098291,0.114630,-0.000307,-0.982917,38
7,3,reduced,total_output_satoshis,0.80,0.6500,0.107153,0.098291,0.114630,-0.000307,-0.982917,32
8,15,full,total_output_satoshis_excl_coinbase,0.80,0.8000,0.107405,0.098501,0.114919,-0.000700,-0.982983,39
9,15,reduced,total_output_satoshis_excl_coinbase,0.85,0.8000,0.107405,0.098501,0.114919,-0.000700,-0.982983,29


In [14]:
seeds = [1, 3, 15, 17, 25, 29, 30, 36, 42, 50, 51, 67, 100]
diff_seed_results_df = []
for seed in seeds:
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "full"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "reduced"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis", "mean_of_ratios", "full"))
    diff_seed_results_df.append(preprocess_and_test(df, seed, "total_output_satoshis", "mean_of_ratios", "reduced"))

diff_seed_final_results = pd.concat(diff_seed_results_df, ignore_index=True)
diff_seed_final_results

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,1,full,total_output_satoshis_excl_coinbase,0.25,0.5625,0.575491,0.417039,27766.783956,0.062137,0.094429,34
1,1,reduced,total_output_satoshis_excl_coinbase,0.45,0.5000,0.575491,0.417039,27766.783956,0.062137,0.094429,41
2,1,full,total_output_satoshis,0.60,0.5125,0.318122,0.273580,0.450543,0.112502,0.034235,36
3,1,reduced,total_output_satoshis,0.50,0.5000,0.318122,0.273580,0.450543,0.112502,0.034235,22
4,3,full,total_output_satoshis_excl_coinbase,0.50,0.5000,0.575491,0.417039,27766.783956,0.062137,0.094429,22
5,3,reduced,total_output_satoshis_excl_coinbase,0.60,0.5250,0.575491,0.417039,27766.783956,0.062137,0.094429,50
6,3,full,total_output_satoshis,0.50,0.4750,0.318122,0.273580,0.450543,0.112502,0.034235,22
7,3,reduced,total_output_satoshis,0.50,0.4500,0.318122,0.273580,0.450543,0.112502,0.034235,42
8,15,full,total_output_satoshis_excl_coinbase,0.50,0.5000,0.575491,0.417039,27766.783956,0.062137,0.094429,22
9,15,reduced,total_output_satoshis_excl_coinbase,0.65,0.5125,0.575491,0.417039,27766.783956,0.062137,0.094429,36
